# Basira — démonstration du pipeline (DEMO MODE)

Ce notebook exécute, sur le clip d'exemple **synthétique et libre de droits** (`samples/sample_medina.mp4`, CC0),
les étapes clés du pipeline : extraction audio → détection de la parole (VAD) → transcription →
calcul des silences → images clés → brouillons d'audiodescription → WebVTT.

Par défaut il tourne **sans réseau ni clé d'API** (`ASR_PROVIDER=demo`, `VLM_PROVIDER=demo`) :
les brouillons proviennent d'un script de référence et sont étiquetés `demo`.
Passez `ASR_PROVIDER=whisper` pour une vraie transcription faster-whisper (téléchargement du modèle).

In [1]:
import os, sys, tempfile
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
os.environ.setdefault("ASR_PROVIDER", "demo")
os.environ.setdefault("VLM_PROVIDER", "demo")
os.environ.setdefault("DATA_DIR", tempfile.mkdtemp())

from app.config import get_settings
from app.pipeline import media
from app.pipeline.vad import detect_speech, compute_gaps, merge_intervals, overlaps
from app.pipeline.fit import fit_to_gap, max_words, estimate_seconds
from app.pipeline.export import Cue, to_vtt
from app.providers import asr
from app.providers.vision import get_vision, DraftRequest

settings = get_settings()
VIDEO = ROOT / "samples" / "sample_medina.mp4"
WORK = Path(tempfile.mkdtemp())
meta = media.probe(VIDEO)
meta

{'duration': 36.0,
 'width': 854,
 'height': 480,
 'fps': 24.0,
 'video_codec': 'h264',
 'audio_codec': 'aac',
 'has_audio': True}

## 1. Audio + détection de la parole (Silero VAD, repli webrtcvad)

In [2]:
wav = media.extract_audio(VIDEO, WORK / "audio.wav")
speech, engine = detect_speech(wav, settings.vad_engine)
print(engine, [(round(a, 2), round(b, 2)) for a, b in speech])

C:\Users\louay\OneDrive\Bureau\hackathonjunior\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


silero [(2.91, 3.68), (3.9, 5.57), (11.9, 14.15), (20.96, 23.27), (29.95, 32.48)]


## 2. Transcription

In [3]:
res = asr.transcribe(settings, wav, "fr", ROOT / "samples" / "sample.transcript.json")
print(res.provider, res.model, res.notes)
for s in res.segments:
    print(f"{s.start:6.2f}–{s.end:6.2f}  {s.text}")

demo sidecar-transcript ["DEMO : transcription de référence du clip d'exemple (pas de modèle ASR)."]
  3.00–  5.81  Ahla bik ! Bienvenue à Sidi Bou Saïd.
 12.00– 14.44  Le café des Nattes est juste en haut de la rue.
 21.00– 23.47  Deux thés à la menthe, s'il vous plaît.
 30.00– 32.73  Merci beaucoup. À demain, inchallah.


## 3. Silences exploitables (≥ 1,5 s, marge de 0,25 s autour de la parole)

In [4]:
asr_only = [(s.start, s.end) for s in res.segments if not overlaps((s.start, s.end), speech, 0.0)]
speech_all = merge_intervals(speech + asr_only, join_within=0.1)
gaps = compute_gaps(speech_all, meta["duration"], settings.min_gap_seconds, settings.gap_padding_seconds)

def bar(t0, t1, ch, width=72, dur=meta["duration"]):
    return int(t0 / dur * width), max(1, int((t1 - t0) / dur * width)), ch
line = [" "] * 72
for a, b in speech_all:
    s, n, _ = bar(a, b, "#"); line[s:s + n] = "#" * n
for a, b in gaps:
    s, n, _ = bar(a, b, "."); line[s:s + n] = "." * n
print("|" + "".join(line) + "|   # parole   . silence exploitable")
gaps

|.....# ### ........... #### ............ ####  ............##### ...... |   # parole   . silence exploitable


[(0.0, 2.658),
 (5.822, 11.65),
 (14.398, 20.706),
 (23.518, 29.698),
 (32.734, 36.0)]

## 4. Images clés (seulement dans les silences + changements de plan)

In [5]:
from app.service import select_keyframe_times
cuts = media.detect_scene_changes(VIDEO, settings.scene_threshold)
plan = {g: select_keyframe_times(g[0], g[1], cuts, settings.max_keyframes_per_gap) for g in gaps}
print("changements de plan :", cuts)
for g, frames in plan.items():
    print(g, frames)

changements de plan : [18.0]
(0.0, 2.658) [(1.33, 'gap')]
(5.822, 11.65) [(6.32, 'gap'), (8.74, 'gap'), (11.15, 'gap')]
(14.398, 20.706) [(14.9, 'gap'), (18.2, 'scene_change'), (20.21, 'gap')]
(23.518, 29.698) [(24.02, 'gap'), (26.61, 'gap'), (29.2, 'gap')]
(32.734, 36.0) [(34.37, 'gap')]


## 5. Brouillons d'audiodescription (VLM) ajustés à la durée disponible

In [6]:
vision = get_vision(settings)
cues = []
for (a, b), frames in plan.items():
    paths = [media.extract_frame(VIDEO, t, WORK / f"f_{t:.2f}.jpg") for t, _ in frames]
    req = DraftRequest(start=a, end=b, language="fr", max_words=max_words(b - a, "fr"),
                       frames=paths, frame_times=[t for t, _ in frames], is_sample=True,
                       scene_change=any(r == "scene_change" for _, r in frames))
    d = vision.draft(req)
    text, flags = fit_to_gap(d.text, b - a, "fr")
    review = d.confidence < settings.confidence_threshold
    print(f"[{a:5.2f}-{b:5.2f}] conf={d.confidence:.2f} {'⚠ À RELIRE ' if review else ''}{flags} ({d.provider}) {text}")
    assert estimate_seconds(text, "fr") <= b - a or "too_long" in flags
    cues.append(Cue(a, min(b, a + estimate_seconds(text, "fr") + 0.4), text, "fr", "ai", "(non validé)"))

[ 0.00- 2.66] conf=0.86 [] (demo) Une porte bleue cloutée, mur chaulé.


[ 5.82-11.65] conf=0.78 [] (demo) Une ruelle ocre au soleil couchant. Une silhouette claire avance vers la droite.


[14.40-20.71] conf=0.45 ⚠ À RELIRE [] (demo) La silhouette sort. Café sombre : une lampe jaune éclaire une table vide.


[23.52-29.70] conf=0.82 [] (demo) Deux verres de thé sont posés sur la table. Fondu au noir.


[32.73-36.00] conf=0.88 [] (demo) Nuit étoilée sur les toits.


## 6. WebVTT (`kind=descriptions`)

Dans l'application, **seules les lignes validées par un humain** sont exportées.
Ici on affiche les brouillons pour illustration.

In [7]:
print(to_vtt(cues, "Démo notebook", demo_mode=vision.name == "demo"))

WEBVTT - Audiodescription

NOTE
Titre : Démo notebook
Piste de type kind=descriptions. Audiodescription assistée par IA, validée par un·e humain·e.
DEMO MODE : les brouillons IA proviennent d'un fournisseur simulé (aucun modèle réel).

NOTE origine=IA puis validée; validée par (non validé)

ad-1
00:00:00.000 --> 00:00:02.658
Une porte bleue cloutée, mur chaulé.

NOTE origine=IA puis validée; validée par (non validé)

ad-2
00:00:05.822 --> 00:00:11.522
Une ruelle ocre au soleil couchant. Une silhouette claire avance vers la droite.

NOTE origine=IA puis validée; validée par (non validé)

ad-3
00:00:14.398 --> 00:00:19.718
La silhouette sort. Café sombre : une lampe jaune éclaire une table vide.

NOTE origine=IA puis validée; validée par (non validé)

ad-4
00:00:23.518 --> 00:00:28.838
Deux verres de thé sont posés sur la table. Fondu au noir.

NOTE origine=IA puis validée; validée par (non validé)

ad-5
00:00:32.734 --> 00:00:35.354
Nuit étoilée sur les toits.

